# Epsilon Fund — CPCV Validation (BB Breakout)

Combinatorial Purged Cross-Validation via `infrastructure/walkforward/cpcv_engine.py`.

Run **after** walk-forward validation to get an unbiased distribution of strategy performance
across all possible train/test splits — not just one rolling path.

---
### What CPCV gives you

| Output | Meaning |
|--------|--------|
| **Path distribution** | 105 complete equity paths (N=8, k=2) — each covers the full dataset once |
| **Mean / Std Sharpe** | Is the strategy reliably positive, or just occasionally lucky? |
| **Parameter stability** | Does the optimizer converge to similar values across all 28 splits? |
| **Tercile comparison** | Do parameter choices in top-performing splits differ from poor ones? |
| **Split heatmap** | Which time periods are structurally hard for this strategy? |

---
### Workflow

1. Complete walk-forward validation for this asset first.
2. Copy `strategy_fn`, `PARAM_DEFS`, and `FIXED_PARAMS` from the walk-forward notebook into the cells below.
3. Set `WF_SHARPE` to the combined OOS Sharpe from walk-forward (for comparison annotation).
4. Run all cells top-to-bottom.
5. Save the results pickle for portfolio-level analysis.

---
### Interpreting results

| Signal | Meaning | Action |
|--------|---------|-------|
| Mean path Sharpe ≈ WF Sharpe, low std | Consistent edge across all splits | High confidence — proceed |
| Mean path Sharpe positive but high std | Strategy works but period-sensitive | Review split heatmap for weak regimes |
| Many negative-Sharpe paths | Edge is concentrated in lucky folds | Revisit strategy architecture |
| CV < 0.10 across most free params | Parameters are stable, not noise-fitted | Safe to fix more params and re-run WF |
| High tercile separation on a param | That param drives performance | Consider narrowing its range or fixing it |

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # <- Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from binance_client import get_binance_client, get_data
from cpcv_engine import (
    run_cpcv,
    cpcv_parameter_analysis,
    cpcv_print_param_suggestions,
    cpcv_summary,
    cpcv_confidence_intervals,
    cpcv_ci_summary,
)
from cpcv_visualizer import (
    plot_cpcv_results,
    plot_parameter_distributions,
    plot_parameter_correlation_matrix,
    plot_split_performance_heatmap,
    plot_tercile_comparison,
)

---
## Data

**Use the same pair, interval, and lookback as the corresponding walk-forward notebook**
so the dataset is identical.

All BB breakout walk-forward notebooks use `INTERVAL = '1h'` and `LOOKBACK = 2151` days.
Do not change these when copying to a per-asset notebook — only change `SYMBOL`.

In [ ]:
SYMBOL   = 'POLUSDT'
INTERVAL = '1h'
LOOKBACK = 2200   # days → ~48 000 1H bars  (TRAIN=17520 + TEST=4380 = ~6 folds)


client   = get_binance_client()
# 1. Fetch the "New" data
df_pol = get_data(client, 'POLUSDT', INTERVAL, LOOKBACK)

# 2. Fetch the "Old" data (MATIC)
# Using the same lookback ensures you catch the full history available
df_matic = get_data(client, 'MATICUSDT', INTERVAL, LOOKBACK)

# 3. Continuity-adjust MATIC prices so the close-to-close transition at the
#    rebrand boundary is exact (standard split/symbol-change adjustment).
#    Binance paused trading for ~3 days during the rebrand; without rescaling,
#    Close.pct_change() at the boundary shows a phantom +8% return that
#    propagates into the CPCV equity curve and across the boundary into ATR /
#    BB / SMA windows.  Multiplying MATIC OHLC by `factor` preserves all
#    percentage returns inside each segment and eliminates the boundary bias.
df_matic = df_matic[df_matic.index < df_pol.index[0]]
factor   = float(df_pol['Close'].iloc[0]) / float(df_matic['Close'].iloc[-1])
for _col in ('Open', 'High', 'Low', 'Close'):
    df_matic[_col] = df_matic[_col] * factor

# 4. Combine — MATIC first, POL second
df = pd.concat([df_matic, df_pol])

# 5. Clean up — drop any duplicate timestamps (keep POL on overlap)
df = df[~df.index.duplicated(keep='last')]

print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
print(f'Stitched {len(df_matic)} legacy MATIC bars (factor={factor:.4f}) before {df_pol.index[0]}')
df.tail(3)

# ── CPCV configuration ────────────────────────────────────────────────────────
N          = 8      # groups to partition data into  ->  C(8,2) = 28 splits
K          = 2      # groups held out per split
N_TRIALS   = 400    # Optuna trials per split (match walk-forward N_TRIALS)
BURNIN     = 200    # bars prepended before each test group for indicator warmup
            # 200 bars = walk-forward BURNIN_BARS — covers max bb_period x4 on resampled 4H
COST       = 0.001  # per-leg trading cost (match walk-forward COST)
PURGE_BARS = 1      # training bars removed at each train/test boundary

# Bar count context:
# LOOKBACK=2151 days at 1h bars -> ~51,600 bars total
# N=8 groups -> ~6,450 bars/group -> ~269 days (~9 months) per group
# Each CPCV path stitches N/K=4 test groups -> covers all ~51,600 bars once

# ── walk-forward Sharpe for comparison annotation (optional) ──────────────────
WF_SHARPE  = None   # <- set to combined OOS Sharpe from walk-forward notebook


---
## Strategy, PARAM_DEFS, and FIXED_PARAMS

> **Paste `strategy_fn`, `PARAM_DEFS`, and `FIXED_PARAMS` from the corresponding
> walk-forward notebook (`bb_breakout_wf/{ASSET}.ipynb`). These must match exactly —
> same function, same fixing decisions, same search ranges.**

The CPCV engine passes the same `(df_slice, params)` interface as the walk-forward engine.
`strategy_fn` must return `(strategy_df, indicator_cols)`.  
Do not change the function signature or indicator columns.

**Important:** The BB breakout strategy resamples 1H data to 4H internally — this is
handled inside `my_strategy` and requires no changes here. The BURNIN=200 bars
ensures the 4H resampling warmup is satisfied (200 1H bars = 50 4H bars) before any
test group begins.

In [30]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
PARAM_DEFS = {
    # ── Bollinger Bands (bb_std removed — cancels in expansion ratio) ──────────
    'bb_period':        ('int',        28,      34),
    'bb_exp_window':      ('int',        18,      20),

    # ── ATR ────────────────────────────────────────────────────────────────────
    'atr_period':  ('int',        12,      16),

    # ── Breakout filter: percentile replaces fixed ATR multiple ───────────────
    # big candle = range > rolling quantile(breakout_pct) over breakout_lookback
    'breakout_pct':     ('float',  0.5193,  0.5569),
    'breakout_lookback': ('int',   20, 100),  # 4H bars for quantile window

    # ── MA periods decoupled: 4H trend filter vs 1H pullback anchor ───────────
    'h4_ma_period':      ('int',   10, 50),
    'slope_epsilon':     ('float', 0.0, 0.003), # allow slight negative slope (normalised by price)
    'h1_ma_period':      ('int',   5,  17),

    # ── Pullback filter split: entry zone and overshoot expiry independent ─────
    'entry_zone_bps':    ('int',   5,  100),   # P1 zone width around 1H SMA
    'overshoot_bps':      ('int',        95,     130),

    # ── Other 1H filters ──────────────────────────────────────────────────────
    'max_1h_bars':       ('int',   12, 48),
    'pullback_atr_mult':  ('float',   1.215,   1.359),
    'trail_atr_mult':   ('float',   3.234,   3.724),

    'adx_period':        ('int',   7, 21),     # fixed at 14 — in PARAM_DEFS for wf_engine visibility
    # ── ADX regime classifier ─────────────────────────────────────────────────
    # adx_strong: above this = strong trend → in_strong_bull active, shorts need -DI > +DI
    # adx_min:    below this = ranging noise → block shorts (no directional edge)
    'adx_strong':        ('float', 20.0, 60.0), # ADX threshold for strong trend

    # ── Regime-conditional TP ─────────────────────────────────────────────────
    # Strong bull (price > trend MA, ADX strong, +DI > -DI): pure trailing stop
    # Otherwise: tp_mult:1 TP cap — protect capital in ranging/bear regimes

    # ── Macro regime gate: long-term MA period (fixed at 200 via FIXED_PARAMS) ──
    'trend_ma_period':  ('int',       178,     251),

    # ── Position sizing ────────────────────────────────────────────────────────
    'risk_per_trade':    ('float', 0.01, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
}

# ── fixed params ──────────────────────────────────────────────────────────────
FIXED_PARAMS = {
    'max_leverage':      2.5,
    'risk_per_trade':    0.02,     

    'bb_period':        30,
    'breakout_pct':     0.5277,
    'trail_atr_mult':   3.407,
    'trend_ma_period':  193,

    'atr_period':         14,
    'bb_exp_window':      20,
    'overshoot_bps':      116,
    'pullback_atr_mult':  1.246,

}

In [4]:
# ── strategy function ─────────────────────────────────────────────────────────
# Paste from walk-forward notebook — must be identical.
# Required signature: my_strategy(df_slice: pd.DataFrame, params: dict)
# Required return:    (strategy_df, indicator_cols)

def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()
    h1 = df.rename(columns=str.lower)

    # ── Resample 1H → 4H ──────────────────────────────────────────────────────
    h4 = h1.resample('4h').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum',
    }).dropna()

    # ── Indicator helpers ──────────────────────────────────────────────────────
    def _atr(d, period):
        hi, lo, cl = d['high'], d['low'], d['close']
        prev_cl = cl.shift(1)
        tr = pd.concat([(hi - lo), (hi - prev_cl).abs(), (lo - prev_cl).abs()], axis=1).max(axis=1)
        return tr.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()

    def _sma(d, period):
        return d['close'].rolling(period).mean()

    def _bb_width(d, period):
        mid = d['close'].rolling(period).mean()
        std = d['close'].rolling(period).std()
        return (std * 2) / mid.replace(0, np.nan)

    def _ma_slope(d, period):
        ma = d['close'].rolling(period).mean()
        return ma - ma.shift(1)

    def _candle_range(d):
        return d['high'] - d['low']

    def _adx(d, period):
        hi, lo, cl = d['high'], d['low'], d['close']
        prev_hi = hi.shift(1)
        prev_lo = lo.shift(1)
        prev_cl = cl.shift(1)
        tr = pd.concat([(hi - lo), (hi - prev_cl).abs(), (lo - prev_cl).abs()], axis=1).max(axis=1)
        plus_dm  = (hi - prev_hi).clip(lower=0).where((hi - prev_hi) > (prev_lo - lo), 0.0)
        minus_dm = (prev_lo - lo).clip(lower=0).where((prev_lo - lo) > (hi - prev_hi), 0.0)
        alpha = 1.0 / period
        atr_w    = tr.ewm(alpha=alpha,      min_periods=period, adjust=False).mean()
        plus_di  = 100 * plus_dm.ewm(alpha=alpha,  min_periods=period, adjust=False).mean() / atr_w.replace(0, np.nan)
        minus_di = 100 * minus_dm.ewm(alpha=alpha, min_periods=period, adjust=False).mean() / atr_w.replace(0, np.nan)
        dx  = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
        adx = dx.ewm(alpha=alpha, min_periods=period, adjust=False).mean()
        return adx, plus_di, minus_di


    # ── 4H indicators ─────────────────────────────────────────────────────────
    h4_atr   = _atr(h4, params['atr_period'])
    h4_range = _candle_range(h4)
    h4_bw    = _bb_width(h4, params['bb_period'])
    h4_slope = _ma_slope(h4, params['h4_ma_period'])

    h4_green = h4['close'] > h4['open']
    h4_red   = h4['close'] < h4['open']

    brk_threshold = h4_range.rolling(int(params['breakout_lookback'])).quantile(params['breakout_pct'])
    big           = h4_range > brk_threshold

    two_big_green = big & big.shift(1) & h4_green & h4_green.shift(1)
    two_big_red   = big & big.shift(1) & h4_red   & h4_red.shift(1)

    bb_exp        = h4_bw > h4_bw.rolling(params['bb_exp_window']).mean()

    h4_slope_norm = h4_slope / h4['close'].replace(0, np.nan)
    slope_eps     = params['slope_epsilon']
    h4_long  = two_big_green & bb_exp & (h4_slope_norm >= -slope_eps)
    h4_short = two_big_red   & bb_exp & (h4_slope_norm <=  slope_eps)

    h4_adx, h4_plus_di, h4_minus_di = _adx(h4, params['adx_period'])

    # ── 1H indicators ─────────────────────────────────────────────────────────
    h1_atr      = _atr(h1,  params['atr_period'])
    h1_range    = _candle_range(h1)
    h1_sma      = _sma(h1,  params['h1_ma_period'])
    trend_period = int(params['trend_ma_period'])
    h1_trend_ma  = h1['close'].rolling(trend_period).mean()
    
    h1_pos_size = (
        params['risk_per_trade'] / (h1_atr / h1['close'])
    ).clip(0.1, params['max_leverage'])

    h4_adx_1h      = h4_adx.shift(1).reindex(h1.index,      method='ffill').fillna(0.0)
    h4_plus_di_1h  = h4_plus_di.shift(1).reindex(h1.index,  method='ffill').fillna(0.0)
    h4_minus_di_1h = h4_minus_di.shift(1).reindex(h1.index, method='ffill').fillna(0.0)

    h4_long_1h  = h4_long.shift(1).reindex(h1.index,  method='ffill').fillna(False)
    h4_short_1h = h4_short.shift(1).reindex(h1.index, method='ffill').fillna(False)

    long_setup_fires  = h4_long_1h  & ~h4_long_1h.shift(1).fillna(False)
    short_setup_fires = h4_short_1h & ~h4_short_1h.shift(1).fillna(False)

    # ── Extract numpy arrays ───────────────────────────────────────────────────
    close_arr     = h1['close'].to_numpy()
    sma_arr       = h1_sma.to_numpy()
    range_arr     = h1_range.to_numpy()
    atr_arr       = h1_atr.to_numpy()
    pos_size_arr  = h1_pos_size.to_numpy()
    trend_ma_arr  = h1_trend_ma.to_numpy()
    adx_arr       = h4_adx_1h.to_numpy()
    plus_di_arr   = h4_plus_di_1h.to_numpy()
    minus_di_arr  = h4_minus_di_1h.to_numpy()
    long_fire     = long_setup_fires.to_numpy()
    short_fire    = short_setup_fires.to_numpy()

    max_1h_bars       = params['max_1h_bars']
    pullback_atr_mult = params['pullback_atr_mult']
    entry_zone_bps    = params['entry_zone_bps']
    overshoot_bps     = params['overshoot_bps']

    # ── State machine — bar by bar ─────────────────────────────────────────────
    n             = len(h1)
    position      = np.zeros(n, dtype=int)
    position_size = np.ones(n)
    stop_loss     = np.zeros(n)

    setup_active    = False
    setup_direction = 0
    bars_since      = 0

    in_trade        = False
    trade_direction = 0
    trade_stop      = 0.0
    trade_tp        = 0.0
    trade_size      = 1.0

    for i in range(1, n):

        # ── 1. Trade management (Recommendation 1 & 3 implemented here) ──────
        if in_trade:
            close      = close_arr[i]
            h1_at_i    = atr_arr[i]
            trail_mult = params['trail_atr_mult']

            if not np.isnan(h1_at_i):
                if trade_direction == 1:
                    trade_stop = max(trade_stop, close - trail_mult * h1_at_i)
                else:
                    trade_stop = min(trade_stop, close + trail_mult * h1_at_i)

            stop_hit = (
                (trade_direction ==  1 and trade_stop > 0 and close <= trade_stop) or
                (trade_direction == -1 and trade_stop > 0 and close >= trade_stop)
            )
            tp_hit = (
                (trade_direction ==  1 and trade_tp > 0 and close >= trade_tp) or
                (trade_direction == -1 and trade_tp > 0 and close <= trade_tp)
            )

            if stop_hit or tp_hit:
                in_trade = False
                # Recommendation 1: Removed re-entry logic. Setup window is not re-armed.
            else:
                position[i]      = trade_direction
                position_size[i] = trade_size
            continue

        # ── 2. Setup detection ────────────────────────────────────────────────
        if not setup_active:
            if long_fire[i]:
                setup_active, setup_direction, bars_since = True,  1, 0
            elif short_fire[i]:
                trend_ma_i = trend_ma_arr[i]
                adx_i      = adx_arr[i]
                plus_di_i  = plus_di_arr[i]
                minus_di_i = minus_di_arr[i]
                adx_strong = params['adx_strong']
                
                above_ma   = not np.isnan(trend_ma_i) and close_arr[i] > trend_ma_i
                bull_trend = adx_i > adx_strong and plus_di_i > minus_di_i
                if not above_ma and not bull_trend:
                    setup_active, setup_direction, bars_since = True, -1, 0

        if not setup_active:
            continue

        # ── 3. Expiry checks ──────────────────────────────────────────────────
        bars_since += 1
        close  = close_arr[i]
        s_ma   = sma_arr[i]
        h1_rng = range_arr[i]
        h1_at  = atr_arr[i]

        if np.isnan(s_ma) or np.isnan(h1_at) or s_ma == 0:
            continue

        if bars_since > max_1h_bars:
            setup_active = False
            continue

        if h1_rng > pullback_atr_mult * h1_at:
            setup_active = False
            continue

        if setup_direction == 1:
            if close < s_ma - (s_ma * overshoot_bps / 10000):
                setup_active = False
                continue
        else:
            if close > s_ma + (s_ma * overshoot_bps / 10000):
                setup_active = False
                continue

        # ── 4. Entry (Recommendation 2 & 3 implemented here) ─────────────────
        bps_from_sma = abs(close - s_ma) / s_ma * 10000
        in_zone = bps_from_sma <= entry_zone_bps

        prev_close = close_arr[i - 1]
        momentum_ok = (
            (setup_direction ==  1 and close > prev_close) or
            (setup_direction == -1 and close < prev_close)
        )

        if in_zone and momentum_ok:
            raw_size   = pos_size_arr[i]
            sz         = raw_size if not np.isnan(raw_size) else 1.0
            trail_mult = params['trail_atr_mult']

            # Recommendation 3: stop_atr_mult removed. Initial stop is the trail distance.
            initial_stop_dist = trail_mult * h1_at
            ts_val = (close - initial_stop_dist) if setup_direction == 1 else (close + initial_stop_dist)

            # Recommendation 2: Hard-coded TP logic
            in_strong_bull = (
                close > trend_ma_arr[i]
                and adx_arr[i] > params['adx_strong']
                and plus_di_arr[i] > minus_di_arr[i]
            )
            
            if in_strong_bull or np.isnan(trend_ma_arr[i]):
                tp_val = 0.0   # ride the trend via trailing stop only
            else:
                # Set fixed 6:1 TP ratio relative to the initial ATR-based risk
                tp_dist = 6 * initial_stop_dist
                tp_val = (close + tp_dist) if setup_direction == 1 else (close - tp_dist)

            position[i]      = setup_direction
            position_size[i] = sz
            stop_loss[i]     = ts_val

            in_trade        = True
            trade_direction = setup_direction
            trade_stop      = ts_val   
            trade_tp        = tp_val   
            trade_size      = sz
            setup_active    = False

    # ─────────────────────────────────────────────────────────────────────────
    indicator_cols      = ['SMA']
    df['SMA']           = h1_sma.to_numpy()
    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_loss

    return df, indicator_cols

---
## Score and Reject Functions

Paste the `score_fn` and `reject_fn` from the corresponding walk-forward notebook,
or keep the defaults below. The CPCV engine uses the same scoring interface as the
walk-forward engine.

**Note on `reject_fn` thresholds:** Each CPCV test group covers ~269 days (~9 months).
Trade counts per group will be lower than a full WF fold. The default below uses
`num_trades < 5` instead of the WF notebook's `< 15` to avoid over-rejecting sparse
but valid periods.

In [5]:
# ── scoring function ─────────────────────────────────────────────────────────
# Paste from the walk-forward notebook, or keep this default.
# Set SCORE_FN = None to use the engine default.

def SCORE_FN(metrics):
    SHARPE_MAX = 3.5
    CALMAR_MAX = 16.0
    RETURN_MAX = 100.0
    calmar = (metrics['total_return'] / abs(metrics['max_drawdown'])
              if metrics['max_drawdown'] != 0 else 0.0)
    s = np.clip(metrics['sharpe_ratio'] / SHARPE_MAX, 0, 1)
    c = np.clip(calmar                  / CALMAR_MAX, 0, 1)
    r = np.clip(metrics['total_return'] / RETURN_MAX, 0, 1)
    return 0.50 * s + 0.30 * c + 0.20 * r

# ── rejection filter ─────────────────────────────────────────────────────────
# Trials that return True are scored -999 and discarded by Optuna.
# Set REJECT_FN = None to use the engine default.
# Thresholds are looser than the WF notebook because CPCV groups are ~9 months,
# not the full 2-year IS window — fewer trades per group is expected.

def REJECT_FN(metrics):
    if metrics is None:                return True
    if metrics['num_trades']   < 5:    return True
    if metrics['win_rate']     < 0.35: return True
    if metrics['max_drawdown'] < -0.80: return True
    if metrics['profit_factor'] < 0.5: return True
    return False

---
## Run CPCV

Partitions the data into `N` equal groups and optimises on every `C(N, K)` training
complement. Each of the 28 splits produces an OOS evaluation on the `K` held-out groups.
All groups are then stitched into 105 complete equity paths (for N=8, k=2).

| Parameter | Description | Value |
|-----------|-------------|-------|
| `N` | Groups — 8 gives C(8,2)=28 splits and 105 complete paths | `8` |
| `K` | Groups held out per split | `2` |
| `N_TRIALS` | Optuna trials per split — match walk-forward for consistency | `400` |
| `BURNIN` | Bars prepended to each test group for indicator warmup | `200` |
| `PURGE_BARS` | Training bars purged at each train/test boundary | `1` |
| `COST` | Per-leg trading cost fraction | `0.001` |

> **Runtime:** `C(N, K) × N_TRIALS` Optuna evaluations.  
> For N=8, K=2, N_TRIALS=400: 28 × 400 = 11,200 backtests.  
> 1H data is heavier per backtest than daily — expect 30–90 min depending on hardware.

In [31]:
results = run_cpcv(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    N            = N,
    k            = K,
    purge_bars   = PURGE_BARS,
    n_trials     = N_TRIALS,
    burnin       = BURNIN,
    cost         = COST,
    score_fn     = SCORE_FN,
    reject_fn    = REJECT_FN,
    verbose      = True,
)

CPCV: N=8  k=2  splits=28  paths=105  trials=400  burnin=200  purge=1
  Group 1: [2020-04-22 → 2021-01-22]  (6587 bars)
  Group 2: [2021-01-22 → 2021-10-24]  (6587 bars)
  Group 3: [2021-10-24 → 2022-07-25]  (6587 bars)
  Group 4: [2022-07-25 → 2023-04-26]  (6587 bars)
  Group 5: [2023-04-26 → 2024-01-25]  (6587 bars)
  Group 6: [2024-01-25 → 2024-10-29]  (6587 bars)
  Group 7: [2024-10-29 → 2025-07-31]  (6587 bars)
  Group 8: [2025-07-31 → 2026-05-01]  (6587 bars)

Fixed (10): ['max_leverage', 'risk_per_trade', 'bb_period', 'breakout_pct', 'trail_atr_mult', 'trend_ma_period', 'atr_period', 'bb_exp_window', 'overshoot_bps', 'pullback_atr_mult']
Free  (8): ['breakout_lookback', 'h4_ma_period', 'slope_epsilon', 'h1_ma_period', 'entry_zone_bps', 'max_1h_bars', 'adx_period', 'adx_strong']

Split 1/28 done | IS score: 0.80 | OOS groups (0, 1) Sharpe: 1.61, 1.60
Split 2/28 done | IS score: 0.80 | OOS groups (0, 2) Sharpe: 1.27, 1.49
Split 3/28 done | IS score: 0.71 | OOS groups (0, 3) Sharpe

### CPCV Summary — display flags

`cpcv_summary` is split into two cells so the output stays readable. Each flag controls one section of the printed report. Toggle them to focus on what you need.

| Flag | Default | What it prints |
|---|---|---|
| `show_group_legend` | `True` | **Group date ranges** — maps each group number (g1–g8) to its calendar window. |
| `show_distribution` | `True` | **Path distribution table** — Mean / Std / Min / Q25 / Median / Q75 / Max / IQR across all 105 paths. |
| `show_paths` | `False` | **Full per-path table** — one row per path (105 rows). |
| `show_highlights` | `True` | **Top 5 / Bottom 5 paths** — ranked by Sharpe. |
| `show_split_legend` | `False` | **Split legend** — decodes the `gN->sM` notation. |
| `show_ci` | `True` | **Confidence intervals** — overlap-adjusted CI for Sharpe, Calmar, and Max DD. |

In [32]:
# Group legend + path distribution percentile table
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = True,
    show_paths        = False,
    show_highlights   = False,
    show_split_legend = False,
    show_ci           = True,
)


════════════════════════════════════════════════════
PATH DISTRIBUTION  (105 paths)
════════════════════════════════════════════════════
Metric       Sharpe     Calmar     Max DD     Return
──────── ────────── ────────── ────────── ──────────
Mean           1.48       2.78    -27.66%   3283.86%
Std            0.20       0.86      2.82%   2384.40%
Min            1.03       1.29    -33.58%    757.54%
Q25            1.34       2.17    -28.33%   1785.02%
Median         1.45       2.60    -27.20%   2393.24%
Q75            1.59       3.19    -26.20%   3968.90%
Max            2.06       6.00    -21.39%  14197.95%
IQR            0.25       1.02      2.13%   2183.88%

════════════════════════════════════════════════════════════════════════
CONFIDENCE INTERVALS  (95%)
════════════════════════════════════════════════════════════════════════
Metric   Method                      Lower     Upper  Note
───────  ───────────────────────  ────────  ────────  ─────────────────
Sharpe   Naive (N=105)    

In [8]:
# Top/bottom highlights + split legend
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = False,
    show_paths        = False,
    show_highlights   = True,
    show_split_legend = True,
    show_ci           = False,
)


════════════════════════════════════════════
TOP 5 PATHS BY SHARPE
════════════════════════════════════════════
  Rank   Sharpe   Calmar    Max DD    Return
────── ──────── ──────── ───────── ─────────
    #1     1.05     0.87   -54.56%   929.22%
       g1→s4  g2→s8  g3→s8  g4→s22  g5→s4  g6→s26  g7→s26  g8→s22
    #2     0.88     0.71   -52.22%   564.86%
       g1→s2  g2→s10  g3→s2  g4→s22  g5→s10  g6→s26  g7→s26  g8→s22
    #3     0.88     0.59   -53.77%   426.00%
       g1→s1  g2→s1  g3→s15  g4→s22  g5→s15  g6→s26  g7→s26  g8→s22
    #4     0.83     0.67   -45.94%   397.51%
       g1→s4  g2→s12  g3→s16  g4→s22  g5→s4  g6→s16  g7→s12  g8→s22
    #5     0.73     0.54   -50.18%   326.57%
       g1→s4  g2→s8  g3→s8  g4→s20  g5→s4  g6→s20  g7→s28  g8→s28

════════════════════════════════════════════
BOTTOM 5 PATHS BY SHARPE
════════════════════════════════════════════
  Rank   Sharpe   Calmar    Max DD    Return
────── ──────── ──────── ───────── ─────────
    #1    -0.79    -0.38   -92

---
## Path-Level Charts

Three outputs:
- **Equity curves** — all 105 paths (semi-transparent blue), the mean path (bold amber),
  and the min-max envelope. Vertical dotted lines mark group boundaries g0-g7.
- **Sharpe histogram** — distribution of the 105 path-level Sharpe ratios with an
  overlap-adjusted confidence interval overlay.
- **Confidence interval text table** — naive and overlap-adjusted 95% CIs for Sharpe,
  Calmar, and Max DD.

In [33]:
ci = cpcv_confidence_intervals(results)
plot_cpcv_results(results, wf_sharpe=WF_SHARPE, ci_results=ci);

---
## Parameter Analysis

Compute the analysis object once — all four charts below read from it.

In [34]:
analysis = cpcv_parameter_analysis(results)

# Prints consensus_ranges table + copy-pasteable PARAM_DEFS / FIXED_PARAMS blocks.
# CV < 0.10  -> "fix at {median}"   : converges consistently, safe to anchor.
# CV 0.10-0.25 -> "narrow to IQR"  : reduce search range to Q25-Q75 next WF run.
# CV > 0.25  -> "keep current range": period-sensitive, keep free.
cpcv_print_param_suggestions(results, analysis)

                   recommended_low  recommended_high  current_low  current_high              action
param                                                                                              
breakout_lookback        38.750000         80.250000         20.0       100.000  keep current range
h4_ma_period             16.750000         38.250000         10.0        50.000  keep current range
slope_epsilon             0.000856          0.002229          0.0         0.003  keep current range
h1_ma_period              7.000000          8.500000          5.0        17.000  keep current range
entry_zone_bps           32.000000         35.750000          5.0       100.000  keep current range
max_1h_bars              29.250000         44.000000         12.0        48.000  keep current range
adx_period                9.750000         17.250000          7.0        21.000  keep current range
adx_strong               24.362447         49.696471         20.0        60.000  keep current range


### Parameter Distributions

One subplot per free parameter. Each dot is one split's best value, jittered horizontally
and **coloured by that split's OOS Sharpe** (red = low, green = high).

- **IQR band** (shaded blue) — the middle 50% of optimised values across splits.
- **Median line** (amber dashed) — the central tendency.
- **Subplot subtitle** — CV verdict (Stable / Moderate / Scattered) and recommended action.

In [11]:
plot_parameter_distributions(results, analysis=analysis);

### Parameter Correlation Matrix

Pearson correlation between every pair of free parameters across the 28 splits.

- **Red** (r close to -1) — substitutable parameters; consider fixing one.
- **Blue** (r close to +1) — parameters that move together; one may be redundant.
- **White** (r ~= 0) — independent.

Any |r| > 0.6 pair is worth investigating for potential over-parameterisation.

In [12]:
plot_parameter_correlation_matrix(analysis);

### Split Performance Heatmap

An N x N grid where cell (i, j) shows the **OOS Sharpe of the split that held out
groups i and j** for testing.

- **Green** — that group-pair was OOS-profitable.
- **Red** — those two periods were structurally hard OOS.

A consistent block of red around a particular group index means one time period is
structurally difficult regardless of which other period it's paired with. Cross-reference
with the group date legend in `cpcv_summary` to identify that regime.

In [13]:
plot_split_performance_heatmap(results);

### Tercile Comparison

One subplot per free parameter showing **top third vs bottom third of splits by OOS Sharpe**.

| Separation | Interpretation |
|------------|----------------|
| < 0.5 | No meaningful difference — parameter choice doesn't predict split quality |
| 0.5-1.0 | Moderate signal — consider narrowing the range toward the top-tercile median |
| > 1.0 | Strong signal — this parameter's value is predictive of OOS performance |

In [14]:
plot_tercile_comparison(results, analysis);

---
## Save Results

Pickle the full results dict for portfolio-level analysis.

Naming convention: `oos/{SYMBOL.lower()}_{INTERVAL}_cpcv.pkl`  
Example: BTCUSDT at 1h -> `oos/btcusdt_1h_cpcv.pkl`

The portfolio notebook (`portfolio_cpcv.ipynb`) loads these pkl files by asset name.

In [15]:
os.makedirs('oos', exist_ok=True)
results_file = os.path.join('oos', f'{SYMBOL.lower()}_{INTERVAL}_cpcv.pkl')
pd.to_pickle(results, results_file)
print(f'Saved to {results_file}')

Saved to oos/polusdt_1h_cpcv.pkl
